# SnowHealth Demand Forecasting — Model Training

Trains two regression models (XGBoost and LightGBM) to predict `UNITS_SOLD`, using **Snowflake Experiment Tracking** to log every run and the **Snowflake Model Registry** to persist the best model for downstream inference.

| Step | What happens |
|---|---|
| Load features | Read the engineered features table created in notebook 02 |
| Experiment tracking | Initialize a Snowflake experiment to log params, metrics, and artifacts per run |
| Train XGBoost | Baseline XGBRegressor with manual param/metric/artifact logging |
| Train LightGBM | Baseline LGBMRegressor with the same logging pattern |
| Compare & select | Pick the best model by RMSE |
| Register model | Log the winner to the Snowflake Model Registry with `WAREHOUSE` target |
| Lineage | Verify the source-data → model lineage chain |

> Requires `02_feature_engineering.ipynb` to have been run first.

### Initialize Session
Sets up the Python path to `source/`, connects to Snowflake, and imports helper functions from `source/model_utils.py` (metrics computation, feature importance extraction, model comparison) and `source/feature_engineering.py` (feature column identification).

In [ ]:
import sys, os

# Resolve source/ (works from root or notebooks/ subdirectory)
for _p in [os.path.join(os.getcwd(), 'source'), os.path.join(os.getcwd(), '..', 'source')]:
    if os.path.isdir(_p): sys.path.insert(0, os.path.realpath(_p)); break

from snowflake.snowpark.context import get_active_session
import config as cfg
from model_utils import (
    train_and_evaluate, compute_metrics, get_feature_importance,
    save_feature_importance_artifact, compare_models,
)
from feature_engineering import get_feature_columns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_MODELS_SCHEMA}").collect()
print(f"Session ready: {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}")

### Load Features and Prepare Train/Test Split
Reads the engineered features table from notebook 02. The `get_feature_columns()` helper automatically identifies all numeric columns, excluding IDs, timestamps, and the target. Data is split 80/20 for training and evaluation, using a fixed random seed for reproducibility.

In [ ]:
df = session.table(cfg.SNOWHEALTH_FEATURE_TABLE).to_pandas()
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")

feature_cols = get_feature_columns(df, cfg.TARGET_COLUMN)
print(f"Features: {len(feature_cols)} numeric columns")

X = df[feature_cols].astype(float)
y = df[cfg.TARGET_COLUMN].astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=cfg.TEST_SIZE, random_state=cfg.RANDOM_STATE)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

### Initialize Snowflake Experiment Tracking
**Experiment Tracking** is Snowflake's built-in MLflow-compatible system for organizing training runs. Each run captures:
- **Parameters** — hyperparameters and configuration (stored as strings)
- **Metrics** — evaluation scores (MAE, RMSE, MAPE, R²)
- **Artifacts** — files like feature importance JSON
- **Models** — the trained model object itself (via `log_model`)

All runs are grouped under the experiment name defined in `config.py` and can be compared in **Snowsight → AI & ML → Experiments**.

In [ ]:
from snowflake.ml.experiment import ExperimentTracking

exp = ExperimentTracking(
    session=session,
    database_name=cfg.SNOWHEALTH_DATABASE,
    schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA,
)
exp.set_experiment(cfg.SNOWHEALTH_EXPERIMENT_NAME)

# Delete stale runs if re-running this notebook
for run_name in ('xgboost_baseline', 'lightgbm_baseline'):
    try:
        exp.delete_run(run_name)
    except:
        pass
print(f"Experiment: {cfg.SNOWHEALTH_EXPERIMENT_NAME}")

### Train XGBoost Baseline
Trains an `XGBRegressor` with the baseline parameters from `config.py` (200 trees, max_depth=6, learning_rate=0.1). Everything is logged inside a single `exp.start_run()` context:
- `log_params` — all hyperparameters so the run is fully reproducible
- `log_metrics` — MAE, RMSE, MAPE, R² computed by `model_utils.compute_metrics()`
- `log_artifact` — feature importance JSON saved to `/tmp` and attached to the run

The run automatically closes when the `with` block exits, and a Snowsight URL is printed.

In [ ]:
from xgboost import XGBRegressor

xgb_model, xgb_preds, xgb_metrics = train_and_evaluate(
    XGBRegressor, cfg.XGBOOST_BASELINE_PARAMS,
    X_train, y_train, X_test, y_test,
    fit_kwargs={'eval_set': [(X_test, y_test)], 'verbose': False},
)

with exp.start_run('xgboost_baseline'):
    exp.log_params(cfg.XGBOOST_BASELINE_PARAMS)
    exp.log_param('model_type', 'XGBRegressor')
    exp.log_param('n_features', len(feature_cols))
    exp.log_param('train_size', len(X_train))
    exp.log_metrics(xgb_metrics)

    xgb_fi = get_feature_importance(xgb_model, feature_cols, 'xgboost')
    xgb_fi_path = save_feature_importance_artifact(xgb_fi, 'xgboost_baseline')
    exp.log_artifact(xgb_fi_path, artifact_path='feature_importance')

print('XGBoost Baseline:')
for k, v in xgb_metrics.items():
    print(f'  {k}: {v}')

### Train LightGBM Baseline
Same pattern as XGBoost but with LightGBM's `LGBMRegressor`. LightGBM tends to be faster and often matches or beats XGBoost on structured tabular data. Both runs are logged under the same experiment for side-by-side comparison.

In [ ]:
from lightgbm import LGBMRegressor

lgbm_model, lgbm_preds, lgbm_metrics = train_and_evaluate(
    LGBMRegressor, cfg.LIGHTGBM_BASELINE_PARAMS,
    X_train, y_train, X_test, y_test,
    fit_kwargs={'eval_X': X_test, 'eval_y': y_test},
)

with exp.start_run('lightgbm_baseline'):
    exp.log_params(cfg.LIGHTGBM_BASELINE_PARAMS)
    exp.log_param('model_type', 'LGBMRegressor')
    exp.log_param('n_features', len(feature_cols))
    exp.log_param('train_size', len(X_train))
    exp.log_metrics(lgbm_metrics)

    lgbm_fi = get_feature_importance(lgbm_model, feature_cols, 'lightgbm')
    lgbm_fi_path = save_feature_importance_artifact(lgbm_fi, 'lightgbm_baseline')
    exp.log_artifact(lgbm_fi_path, artifact_path='feature_importance')

print('LightGBM Baseline:')
for k, v in lgbm_metrics.items():
    print(f'  {k}: {v}')

### Compare Models
Uses `model_utils.compare_models()` to build a side-by-side comparison table sorted by RMSE. This is the same data that Snowflake Experiment Tracking stores — you can also view it interactively in **Snowsight → AI & ML → Experiments → SNOWHEALTH_DEMAND_FORECAST_EXPERIMENT**.

In [ ]:
results = {
    "XGBoost Baseline": xgb_metrics,
    "LightGBM Baseline": lgbm_metrics,
}
comparison_df = compare_models(results)
print(comparison_df.to_string(index=False))

### Feature Importance Visualization
Plots the top 15 most important features for each model side by side. Feature importance helps validate that the model is learning meaningful patterns (e.g., rolling averages and lag features should rank highly for a time-series demand problem).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

xgb_top = xgb_fi.head(15)
axes[0].barh(range(len(xgb_top)), xgb_top['Importance_Pct'].values, color='#29B5E8')
axes[0].set_yticks(range(len(xgb_top)))
axes[0].set_yticklabels(xgb_top['Feature'].values)
axes[0].set_xlabel('Importance (%)')
axes[0].set_title('XGBoost — Top 15 Features', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()

lgbm_top = lgbm_fi.head(15)
axes[1].barh(range(len(lgbm_top)), lgbm_top['Importance_Pct'].values, color='#333333')
axes[1].set_yticks(range(len(lgbm_top)))
axes[1].set_yticklabels(lgbm_top['Feature'].values)
axes[1].set_xlabel('Importance (%)')
axes[1].set_title('LightGBM — Top 15 Features', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

plt.suptitle('Feature Importance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### Register Best Model to Snowflake Model Registry
The **Model Registry** is Snowflake's versioned model store. `registry.log_model()` serializes the trained model object, its input/output signature, and metadata (metrics, comments) into a Snowflake MODEL object.

- **`target_platforms=["WAREHOUSE"]`** — the model will run inference via SQL on a standard warehouse (no SPCS needed)
- **`signatures`** — defines the input/output schema so Snowflake can validate predictions at runtime
- **`sample_input_data`** — used to infer column types and validate the signature
- **`metrics`** — attached to the version for comparison in the registry UI

The best model (lowest RMSE) is automatically selected. This establishes the **lineage link** from source features → model.

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model.model_signature import infer_signature

registry = Registry(session=session, database_name=cfg.SNOWHEALTH_DATABASE, schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA)

# Pick the best model by RMSE
if lgbm_metrics['rmse'] <= xgb_metrics['rmse']:
    best_model, best_name, best_metrics = lgbm_model, 'LightGBM', lgbm_metrics
else:
    best_model, best_name, best_metrics = xgb_model, 'XGBoost', xgb_metrics

sig = infer_signature(X_train.head(10), y_train.head(10))

# Drop existing model if re-running (remove notebook 05's gateway/services first;
# a model can't be dropped while SPCS inference services use it)
_models = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}"
session.sql(f"DROP GATEWAY IF EXISTS {_models}.SNOWHEALTH_GATEWAY").collect()
for _svc in ['SNOWHEALTH_CHAMPION_SVC', 'SNOWHEALTH_CHALLENGER_SVC']:
    session.sql(f"DROP SERVICE IF EXISTS {_models}.{_svc}").collect()
session.sql(f"DROP MODEL IF EXISTS {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.{cfg.SNOWHEALTH_MODEL_NAME}").collect()

mv = registry.log_model(
    best_model,
    model_name=cfg.SNOWHEALTH_MODEL_NAME,
    version_name="baseline_v1",
    signatures={"predict": sig},
    comment=f"SnowHealth demand forecast baseline — {best_name}",
    metrics=best_metrics,
    sample_input_data=X_train.head(10),
    target_platforms=["WAREHOUSE"],
    options={"enable_explainability": True},  # adds explain() for SHAP in notebook 05
)
print(f"Registered: {cfg.SNOWHEALTH_MODEL_NAME}/baseline_v1 ({best_name})")
print(f"Metrics: {best_metrics}")

### Verify Registration and Test Inference
Confirms the model is in the registry, lists its versions, and runs a small warehouse-based prediction to prove the end-to-end pipeline works. The `mv.run()` call executes the model inside the Snowflake warehouse — no external compute needed.

In [ ]:
model_ref = registry.get_model(cfg.SNOWHEALTH_MODEL_NAME)
for v in model_ref.versions():
    print(f"  Version: {v.version_name} — {v.comment}")

test_pred = mv.run(X_test.head(5), function_name="predict")
print(f"\nSample predictions (first 5 rows):")
print(test_pred)

### Lineage Verification
Snowflake automatically tracks the **data lineage** chain: source tables → feature tables → models. When a model is registered via `registry.log_model()` with `sample_input_data`, Snowflake captures which tables contributed to the model's training data.

This cell queries the lineage graph to confirm the model is linked back to the features table. Lineage metadata propagates asynchronously and may take up to 2 hours to appear in `ACCOUNT_USAGE` views. You can also view it in **Snowsight → Data → Lineage** tab.

In [ ]:
model_fqn = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.{cfg.SNOWHEALTH_MODEL_NAME}"

try:
    lineage_df = session.sql(f"""
        SELECT SOURCE_OBJECT_NAME, SOURCE_OBJECT_DOMAIN,
               TARGET_OBJECT_NAME, TARGET_OBJECT_DOMAIN
        FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE('{model_fqn}', 'MODEL', 'UPSTREAM', 5))
        LIMIT 20
    """).to_pandas()
    if len(lineage_df) > 0:
        print(f"Upstream lineage for {cfg.SNOWHEALTH_MODEL_NAME}: {len(lineage_df)} edge(s)")
        print(lineage_df.to_string(index=False))
    else:
        print("No lineage edges yet — metadata propagates asynchronously (up to 2 hours).")
except Exception as e:
    print(f"Lineage query note: {type(e).__name__}")
    print("Lineage metadata propagates asynchronously — check Snowsight → Data → Lineage tab.")

### Summary
Recap of everything produced in this notebook. All experiments, model versions, and lineage are accessible in Snowsight.

In [ ]:
print('=' * 60)
print('  MODEL TRAINING COMPLETE')
print('=' * 60)
print(f"  Experiment:  {cfg.SNOWHEALTH_EXPERIMENT_NAME}")
print(f"  Runs:        xgboost_baseline, lightgbm_baseline")
print(f"  Best model:  {best_name} (RMSE={best_metrics['rmse']})")
print(f"  Registered:  {cfg.SNOWHEALTH_MODEL_NAME}/baseline_v1")
print(f"  Target:      WAREHOUSE")
print('=' * 60)
print('  Next → 04_hyperparameter_tuning.ipynb')